# Standard Benchmarks

## নোটবুক পরিচিতি

দুটি স্বাধীন, সম্পূর্ণ স্বয়ং-সম্পূর্ণ demo:

1. **Log-probability-ভিত্তিক multiple-choice স্কোরিং**, হুবহু যেমনটি বাস্তব MMLU/
   HellaSwag evaluation harness-গুলো করে: একটি ছোট character-level mini-GPT
   (Phase 02 Lesson 6-এর একই রেসিপি) "question -> answer" fact-এর toy corpus-এ
   প্রশিক্ষণ দেওয়া হয়। তারপর প্রতিটি toy MMLU-শৈলীর প্রশ্নের জন্য, প্রতিটি
   candidate উত্তরকে মডেলের সেই candidate-এর character-গুলোর মোট
   log-probability দিয়ে (teacher-forced) স্কোর করা হয় এবং argmax বেছে নেওয়া
   হয় — মডেলটিকে কখনো আক্ষরিকভাবে "A"/"B"/"C"/"D" টাইপ করতে বলা হয় না।
   সঠিক উত্তরগুলো training corpus-এ নির্ধারকভাবে (deterministically) বেক করা
   থাকে, তাই ফলে আসা accuracy-টি যাচাইযোগ্যভাবে অর্থবহ (এটি পরীক্ষা করে স্কোরিং
   পদ্ধতি একটি পরিচিত association-কে সঠিকভাবে পুনরুদ্ধার করে কি না — মডেলটি
   "smart" কি না তা নয়)।

2. **pass@k unbiased estimator** (Chen et al., 2021), শূন্য থেকে implement করা
   এবং দুইভাবে sanity-check করা: brute-force Monte Carlo simulation-এর বিরুদ্ধে,
   এবং বড়-n সীমায় textbook সূত্র 1 - (1-p)^k ("kটি i.i.d. Bernoulli trial-এ
   কমপক্ষে একটি সাফল্য")-এর বিরুদ্ধে।

Runtime: CPU-তে ~30-60 সেকেন্ড (ছোট মডেলে 1500 training step)।

## কীভাবে চালাবেন

মূল file-টি হলো `example.py` — `python example.py` দিয়ে চলে। notebook-এ একই
কোড cell-by-cell চালানো হয়; শেষ cell-টি `main()` কল করে।

In [ ]:
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)

## অংশ 1: log-probability multiple-choice স্কোরিং

একটি toy "knowledge base" — প্রতিটি fact একটি (question, correct_answer) জোড়া।
এগুলো নিচের training corpus-এ BAKED (বেক করা) — তাই একটি সঠিক স্কোরিং পদ্ধতির
উচিত এগুলোর প্রতিটিকে নিশ্চিতভাবে পুনরুদ্ধার করা। সাথে মডেলের hyperparameter-ও
এখানে সেট করা হয়।

In [ ]:
# Toy "knowledge base" -- প্রতিটি fact একটি (question, correct_answer) pair।
# এগুলো নিচের training corpus-এ BAKED হয়ে আছে, তাই একটি কার্যকরী স্কোরিং
# পদ্ধতি নিশ্চিতভাবে এদের প্রতিটিই পুনরুদ্ধার করবে।
FACTS = [
    ("what is the capital of france", "paris"),
    ("what is the capital of japan", "tokyo"),
    ("what is the largest planet", "jupiter"),
    ("what is the smallest planet", "mercury"),
    ("what is the chemical symbol for gold", "au"),
    ("who wrote hamlet", "shakespeare"),
]


def fact_sentence(question, answer):
    return f"q: {question}? a: {answer}.\n"


FACT_BLOCK = "".join(fact_sentence(q, a) for q, a in FACTS)
CORPUS = FACT_BLOCK * 40   # পর্যাপ্ত data-র জন্য বারবার — হুবহু Phase 02 Lesson 6-এর মতো

chars = sorted(set(CORPUS))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


def encode(text):
    return [stoi[ch] for ch in text]


data = torch.tensor(encode(CORPUS), dtype=torch.long)

BLOCK_SIZE = 64
D_MODEL = 64
NUM_HEADS = 4
D_FF = 4 * D_MODEL
NUM_LAYERS = 2
BATCH_SIZE = 32
NUM_ITERS = 1500
LEARNING_RATE = 3e-3

## Data batching

প্রশিক্ষণের জন্য প্রতিটি step-এ corpus থেকে random token-সিকোয়েন্সের একটি batch
তোলা হয়: ইনপুট `x` এবং এক-অক্ষর-ডানে-সরানো target `y`।

In [ ]:
def get_batch():
    max_start = len(data) - BLOCK_SIZE - 1
    starts = torch.randint(0, max_start, (BATCH_SIZE,))
    x = torch.stack([data[s:s + BLOCK_SIZE] for s in starts])
    y = torch.stack([data[s + 1:s + 1 + BLOCK_SIZE] for s in starts])
    return x, y

## Mini-GPT: architecture

Phase 02 Lesson 6-এর অভিন্ন রেসিপি (এখানে সংক্ষিপ্ত রাখা): causal
self-attention, feed-forward block, এবং LayerNorm-সহ decoder block-গুলো মিলে
MiniGPT। এর পরবর্তী-character ভবিষ্যদ্বাণীই teacher-forced scoring-এর ভিত্তি।

In [ ]:
# --- Mini-GPT: Phase 02 Lesson 6-এর অভিন্ন রেসিপি (এখানে minimal রাখা হয়েছে) ---

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, block_size):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)).bool())

    def forward(self, x):
        batch, T, d_model = x.shape

        def split_heads(t):
            return t.view(batch, T, self.num_heads, self.d_k).transpose(1, 2)

        Q, K, V = split_heads(self.W_q(x)), split_heads(self.W_k(x)), split_heads(self.W_v(x))
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores.masked_fill(~self.mask[:T, :T], float("-inf"))
        weights = F.softmax(scores, dim=-1)
        out = (weights @ V).transpose(1, 2).contiguous().view(batch, T, d_model)
        return self.W_o(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads, block_size)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList(
            [DecoderBlock(d_model, num_heads, d_ff, block_size) for _ in range(num_layers)]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids, targets=None):
        batch, T = token_ids.shape
        positions = torch.arange(T, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.output_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

## MMLU-শৈলীর স্কোরিং

`score_continuation` হলো বাস্তব MMLU/HellaSwag harness-এর মূল — একটি মাত্র
teacher-forced forward pass দিয়ে continuation-এর মোট log-probability, কখনো
টেক্সট generate করে নয়। `build_mc_questions` প্রতিটি fact-কে একটি toy MMLU-শৈলীর
multiple-choice প্রশ্নে রূপান্তর করে (সঠিক উত্তর + অন্যান্য fact-এর উত্তর থেকে
ধার করা 3টি distractor)।

In [ ]:
@torch.no_grad()
def score_continuation(model, prompt, continuation):
    """বাস্তব MMLU/HellaSwag-শৈলীর harness-এর মূল: মোট log P(continuation | prompt),
    এক মাত্র teacher-forced forward pass দিয়ে হিসাব করা হয়, কখনো টেক্সট generate
    করে নয়।"""
    full_ids = encode(prompt + continuation)
    prompt_len = len(encode(prompt))
    input_ids = torch.tensor([full_ids[:-1]], dtype=torch.long)
    target_ids = full_ids[1:]

    logits, _ = model(input_ids)
    log_probs = F.log_softmax(logits, dim=-1)

    # target_ids[t] হলো সেই token যেটি logits[:, t, :] ভবিষ্যদ্বাণী করে।
    # continuation-এর প্রথম character-এর ভবিষ্যদ্বাণী শুরু হয় t = prompt_len - 1 থেকে।
    total_log_prob = 0.0
    for t in range(prompt_len - 1, len(target_ids)):
        total_log_prob += log_probs[0, t, target_ids[t]].item()
    return total_log_prob


def build_mc_questions():
    """প্রতিটি fact-কে একটি toy MMLU-শৈলীর multiple-choice প্রশ্নে রূপান্তর:
    সঠিক উত্তর, সাথে অন্যান্য fact-গুলোর উত্তর থেকে ধার করা 3টি distractor
    (তাই প্রতিটি option-ই একটি বাস্তব শব্দ যা মডেল দেখেছে, শুধু ভিন্ন প্রশ্নের
    সাথে জোড়া লাগানো -- ঠিক এটিই multiple choice-কে non-trivial করে তোলে)।"""
    all_answers = [a for _, a in FACTS]
    questions = []
    for question, correct in FACTS:
        distractors = [a for a in all_answers if a != correct]
        random.shuffle(distractors)
        options = [correct] + distractors[:3]
        random.shuffle(options)
        questions.append((question, correct, options))
    return questions

## Demo 1: mini-GPT প্রশিক্ষণ ও multiple-choice স্কোরিং

মডেলটিকে training data-তে প্রশিক্ষণ দেওয়া হয়, তারপর প্রতিটি প্রশ্নের জন্য
প্রতিটি candidate-এর মোট log-probability স্কোর করে সর্বোচ্চটি (argmax) নেওয়া
হয় — এবং Toy-MMLU accuracy রিপোর্ট করা হয়।

In [ ]:
def multiple_choice_demo():
    print("=" * 78)
    print("1. LOG-PROBABILITY-BASED MULTIPLE-CHOICE SCORING (how MMLU is really graded)")
    print("=" * 78)
    print(f"Training corpus: {len(FACTS)} facts, repeated into {len(CORPUS)} characters.")
    print(f"Vocabulary: {vocab_size} unique characters.\n")

    model = MiniGPT(vocab_size, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, BLOCK_SIZE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    print("Training the mini-GPT on next-token prediction over the fact corpus...")
    for step in range(1, NUM_ITERS + 1):
        x, y = get_batch()
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 300 == 0 or step == 1:
            print(f"  step {step:5d}  loss = {loss.item():.4f}")

    print("\nEvaluating: for each question, score EVERY candidate answer's total")
    print("log-probability given the shared question prefix, and pick the argmax.\n")

    questions = build_mc_questions()
    num_correct = 0
    for question, correct_answer, options in questions:
        prompt = f"q: {question}? a:"
        scores = {}
        for option in options:
            continuation = f" {option}."
            scores[option] = score_continuation(model, prompt, continuation)

        predicted = max(scores, key=scores.get)
        is_correct = predicted == correct_answer
        num_correct += int(is_correct)

        print(f"  Q: {question}?")
        for option in options:
            marker = " <-- picked" if option == predicted else ""
            gold = " (correct)" if option == correct_answer else ""
            print(f"      log P({option!r} | prompt) = {scores[option]:8.2f}{marker}{gold}")
        print(f"    {'CORRECT' if is_correct else 'WRONG'}\n")

    accuracy = num_correct / len(questions)
    print(f"Toy-MMLU accuracy: {num_correct}/{len(questions)} = {accuracy:.0%}")
    print("\n-> The model was NEVER asked to output a letter or generate free text --")
    print("   every candidate's full continuation was scored by the SAME model in")
    print("   teacher-forced mode, and the highest-log-probability option won. Since")
    print("   the correct associations were literally memorized during training, this")
    print("   accuracy number verifies the SCORING METHOD works correctly (it correctly")
    print("   recovers a known ground truth), which is exactly how lm-evaluation-harness")
    print("   scores real MMLU/HellaSwag questions against real trained LLMs.")


multiple_choice_demo()

## অংশ 2: pass@k unbiased estimator (Chen et al., 2021)

`pass_at_k = 1 - C(n-c, k) / C(n, k)` — n sample-এর একটি random size-k subset-এ
কমপক্ষে একটি সঠিক sample থাকার সম্ভাবনা। `monte_carlo_pass_at_k` brute-force
sanity check: অনেক random subset টেনে empirically fraction মাপা হয়।

In [ ]:
def pass_at_k(n, c, k):
    """1 - P(n sample-এর একটি random size-k subset-এ ZEROটি সঠিক sample থাকে)।"""
    if n - c < k:
        return 1.0   # k-এর চেয়ে কম sample ভুল, তাই যেকোনো subset-এ একটি সঠিক sample থাকবেই
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)


def monte_carlo_pass_at_k(n, c, k, trials=200_000):
    """Brute-force sanity check: n sample থেকে (replacement ছাড়া) সত্যিই random
    size-k subset টানা হয় (c-টি সঠিক চিহ্নিত) এবং অন্তত একটি সঠিক sample-যুক্ত
    subset-এর empirical fraction মাপা হয়।"""
    items = [True] * c + [False] * (n - c)   # True = সঠিক
    successes = 0
    for _ in range(trials):
        subset = random.sample(items, k)
        successes += any(subset)
    return successes / trials


def pass_at_k_demo():
    print("\n" + "=" * 78)
    print("2. THE pass@k UNBIASED ESTIMATOR: pass@k = 1 - C(n-c, k) / C(n, k)")
    print("=" * 78)

    print(f"{'n':>5}{'c':>5}{'k':>5}{'formula':>12}{'monte carlo':>14}")
    cases = [(10, 1, 1), (10, 5, 1), (10, 2, 5), (100, 10, 10), (20, 15, 3)]
    for n, c, k in cases:
        formula_val = pass_at_k(n, c, k)
        mc_val = monte_carlo_pass_at_k(n, c, k, trials=50_000)
        print(f"{n:>5}{c:>5}{k:>5}{formula_val:>12.4f}{mc_val:>14.4f}")

    print("\n-> The closed-form formula and the brute-force Monte Carlo estimate agree")
    print("   to within simulation noise on every case -- confirming the formula really")
    print("   does compute 'probability at least one of a random k-sample subset passed'.")

    print("\n" + "-" * 78)
    print("Sanity check: as n grows with p = c/n held fixed, pass@k should converge to")
    print("the textbook 'at least one success in k i.i.d. Bernoulli(p) trials' formula:")
    print("    1 - (1 - p)^k")
    print("(sampling k out of a large finite pool without replacement behaves more and")
    print("more like k independent draws as the pool size n grows).\n")

    p, k = 0.2, 3
    binomial_limit = 1 - (1 - p) ** k
    print(f"p = c/n = {p}, k = {k}  ->  binomial formula 1-(1-p)^k = {binomial_limit:.4f}\n")
    print(f"{'n':>10}{'c = p*n':>10}{'pass@k (formula)':>20}")
    for n in [10, 100, 1_000, 100_000]:
        c = int(round(p * n))
        print(f"{n:>10}{c:>10}{pass_at_k(n, c, k):>20.4f}")

    print(f"\n-> As n grows (with the same ratio c/n = {p}), the finite-pool pass@k formula")
    print(f"   converges toward the i.i.d.-Bernoulli value {binomial_limit:.4f} -- exactly the")
    print("   sanity check we'd want: pass@k IS 'probability of at least one success,' just")
    print("   computed exactly for a finite, already-drawn sample instead of assuming")
    print("   infinite independent draws.")


pass_at_k_demo()

## সবগুলো demo একসাথে: main()

`main()` দুটি demo একই ক্রমে চালায় — মূল `example.py`-তে এটি
`if __name__ == "__main__":` guard-এর ভেতরে; notebook-এ শেষ cell হিসেবে `main()`
কল করা হয়।

In [ ]:
def main():
    multiple_choice_demo()
    pass_at_k_demo()


main()